# 🧠 Group B — Notebook 2: Build a DNA Transformer

## Main question

> **How does the way we represent DNA change what a Transformer learns and how much computation it needs?**

Group B will build the architecture using understandable PyTorch components.

PyTorch will implement the low-level attention math so that we can focus on **what each model part does**.

## Roadmap

```mermaid
flowchart LR
    A["Same CTCF DNA"] --> B["5 tokenizers"]
    B --> C["Input vectors"]
    C --> D["Position"]
    D --> E["Transformer Encoder"]
    E --> F["Sequence summary"]
    F --> G["Classifier"]
    G --> H["Train + evaluate"]
    H --> I["Compare tokenizers"]
```

# 1. The same biological task

```mermaid
flowchart LR
    A["DNA sequence"] --> B["Custom Transformer"]
    B --> C{"Prediction"}
    C -->|"1"| D["CTCF Binding"]
    C -->|"0"| E["Background"]
```

The biology stays fixed.

Group B changes the **representation and model architecture**.

In [ ]:
# ▶️ RUN — imports and paths

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

DATA_DIR = Path("/global/cfs/cdirs/m4388/projects/project7/ctcf_k562_example")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Dataset:", DATA_DIR)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

### Libraries in this notebook

- `pandas` → readable tables.
- `matplotlib` → graphs.
- `torch` → neural-network pieces and training.
- `sklearn.metrics` → evaluation graphs.

Again, pandas is a **convenience**, not the topic.

In [ ]:
import itertools
import random
import time
import numpy as np

from dataclasses import dataclass
from collections import Counter as PairCounter

from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)


def load_dataset(data_dir):
    """Read DNA and labels and remove invalid/conflicting duplicates."""
    sequences = [
        line.strip().upper()
        for line in (Path(data_dir) / "seqs.txt").read_text().splitlines()
        if line.strip()
    ]

    labels = [
        int(line.strip())
        for line in (Path(data_dir) / "labels.txt").read_text().splitlines()
        if line.strip()
    ]

    label_sets = {}

    for sequence, label in zip(sequences, labels):
        if set(sequence) <= set("ACGT"):
            label_sets.setdefault(sequence, set()).add(label)

    clean_sequences = [
        sequence
        for sequence, values in label_sets.items()
        if len(values) == 1
    ]

    clean_labels = [
        next(iter(label_sets[sequence]))
        for sequence in clean_sequences
    ]

    return clean_sequences, clean_labels


def get_training_sequences(sequences, labels, seed=42):
    """Return the exact training split used later by the model."""
    train_seq, _, _, _ = train_test_split(
        sequences,
        labels,
        test_size=0.20,
        random_state=seed,
        stratify=labels,
    )

    return train_seq


BASE_ID = {
    "A": 1,
    "C": 2,
    "G": 3,
    "T": 4,
}

ONE_HOT = {
    "A": [1., 0., 0., 0.],
    "C": [0., 1., 0., 0.],
    "G": [0., 0., 1., 0.],
    "T": [0., 0., 0., 1.],
}

ALL_6MERS = [
    "".join(chars)
    for chars in itertools.product("ACGT", repeat=6)
]

KMER_ID = {
    token: index + 1
    for index, token in enumerate(ALL_6MERS)
}


class SimpleBPE:
    """Small teaching BPE implementation used behind the scenes."""

    def __init__(self, merges=80):
        self.merges = merges
        self.rules = []
        self.vocab = {
            "<PAD>": 0,
            "<UNK>": 1,
        }

    def _apply_rule(self, tokens, pair, merged):
        output = []
        i = 0

        while i < len(tokens):
            if (
                i < len(tokens) - 1
                and (tokens[i], tokens[i+1]) == pair
            ):
                output.append(merged)
                i += 2
            else:
                output.append(tokens[i])
                i += 1

        return output

    def fit(self, sequences):
        tokenized = [
            list(sequence)
            for sequence in sequences
        ]

        for _ in range(self.merges):
            counts = PairCounter()

            for tokens in tokenized:
                counts.update(
                    zip(tokens[:-1], tokens[1:])
                )

            if not counts:
                break

            pair, _ = counts.most_common(1)[0]
            merged = pair[0] + pair[1]

            self.rules.append(
                (pair, merged)
            )

            tokenized = [
                self._apply_rule(
                    tokens,
                    pair,
                    merged,
                )
                for tokens in tokenized
            ]

        learned_tokens = sorted({
            token
            for tokens in tokenized
            for token in tokens
        })

        self.vocab.update({
            token: index + 2
            for index, token in enumerate(learned_tokens)
        })

    def tokens(self, sequence):
        tokens = list(sequence)

        for pair, merged in self.rules:
            tokens = self._apply_rule(
                tokens,
                pair,
                merged,
            )

        return tokens

    def encode(self, sequence):
        return [
            self.vocab.get(token, 1)
            for token in self.tokens(sequence)
        ]


@dataclass
class TokenizerSpec:
    name: str
    input_kind: str
    vocab_size: int
    max_length: int
    encode: object
    show_tokens: object


def build_tokenizers(train_sequences, bpe_merges=80):
    """Create all five DNA representations."""

    bpe = SimpleBPE(
        merges=bpe_merges
    )

    bpe.fit(
        train_sequences
    )

    tokenizers = {}

    tokenizers["single_nucleotide"] = TokenizerSpec(
        name="single_nucleotide",
        input_kind="token_ids",
        vocab_size=5,
        max_length=200,
        encode=lambda sequence: [
            BASE_ID[base]
            for base in sequence
        ],
        show_tokens=lambda sequence: list(sequence),
    )

    tokenizers["one_hot"] = TokenizerSpec(
        name="one_hot",
        input_kind="one_hot",
        vocab_size=4,
        max_length=200,
        encode=lambda sequence: torch.tensor(
            [
                ONE_HOT[base]
                for base in sequence
            ],
            dtype=torch.float32,
        ),
        show_tokens=lambda sequence: list(sequence),
    )

    tokenizers["overlap_6mer"] = TokenizerSpec(
        name="overlap_6mer",
        input_kind="token_ids",
        vocab_size=4097,
        max_length=195,
        encode=lambda sequence: [
            KMER_ID[sequence[i:i+6]]
            for i in range(len(sequence) - 5)
        ],
        show_tokens=lambda sequence: [
            sequence[i:i+6]
            for i in range(len(sequence) - 5)
        ],
    )

    tokenizers["nonoverlap_6mer"] = TokenizerSpec(
        name="nonoverlap_6mer",
        input_kind="token_ids",
        vocab_size=4097,
        max_length=33,
        encode=lambda sequence: [
            KMER_ID[sequence[i:i+6]]
            for i in range(0, len(sequence) - 5, 6)
        ],
        show_tokens=lambda sequence: [
            sequence[i:i+6]
            for i in range(0, len(sequence) - 5, 6)
        ],
    )

    tokenizers["bpe"] = TokenizerSpec(
        name="bpe",
        input_kind="token_ids",
        vocab_size=len(bpe.vocab),
        max_length=200,
        encode=bpe.encode,
        show_tokens=bpe.tokens,
    )

    return tokenizers, bpe


class EncodedDataset(Dataset):
    """Store already-tokenized DNA examples."""

    def __init__(self, sequences, labels, tokenizer_spec):
        self.examples = []

        for sequence, label in zip(sequences, labels):
            encoded = tokenizer_spec.encode(sequence)

            if not torch.is_tensor(encoded):
                encoded = torch.tensor(
                    encoded,
                    dtype=torch.long,
                )

            self.examples.append(
                (encoded, int(label))
            )

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def collate_batch(batch):
    """Pad variable-length examples so they can form one batch."""

    x_list, y_list = zip(*batch)

    lengths = torch.tensor(
        [len(x) for x in x_list],
        dtype=torch.long,
    )

    padding_value = (
        0.0
        if x_list[0].dtype.is_floating_point
        else 0
    )

    x = pad_sequence(
        x_list,
        batch_first=True,
        padding_value=padding_value,
    )

    positions = torch.arange(
        x.shape[1]
    ).unsqueeze(0)

    padding_mask = (
        positions
        >= lengths.unsqueeze(1)
    )

    return (
        x,
        padding_mask,
        torch.tensor(y_list, dtype=torch.long),
    )


def train_transformer(
    tokenizer_spec,
    model_class,
    sequences,
    labels,
    d_model=64,
    nhead=4,
    layers=2,
    epochs=3,
    batch_size=32,
    learning_rate=1e-3,
    seed=42,
):
    """Train one custom Transformer for one tokenizer."""

    train_seq, val_seq, train_y, val_y = train_test_split(
        sequences,
        labels,
        test_size=0.20,
        random_state=seed,
        stratify=labels,
    )

    train_data = EncodedDataset(
        train_seq,
        train_y,
        tokenizer_spec,
    )

    val_data = EncodedDataset(
        val_seq,
        val_y,
        tokenizer_spec,
    )

    train_loader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_batch,
    )

    val_loader = DataLoader(
        val_data,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_batch,
    )

    model = model_class(
        input_kind=tokenizer_spec.input_kind,
        vocab_size=tokenizer_spec.vocab_size,
        max_length=tokenizer_spec.max_length,
        d_model=d_model,
        nhead=nhead,
        layers=layers,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
    )

    loss_function = nn.CrossEntropyLoss()

    history = {
        "train_loss": [],
        "val_auroc": [],
        "val_auprc": [],
    }

    start_time = time.time()

    for epoch in range(1, epochs + 1):

        # -------- training --------
        model.train()
        training_loss_sum = 0.0

        for x, padding_mask, answers in train_loader:
            x = x.to(DEVICE)
            padding_mask = padding_mask.to(DEVICE)
            answers = answers.to(DEVICE)

            optimizer.zero_grad()

            prediction_scores = model(
                x,
                padding_mask,
            )

            loss = loss_function(
                prediction_scores,
                answers,
            )

            loss.backward()
            optimizer.step()

            training_loss_sum += (
                loss.item()
                * len(answers)
            )

        training_loss = (
            training_loss_sum
            / len(train_data)
        )

        # -------- validation --------
        model.eval()

        true_labels = []
        predicted_labels = []
        probabilities = []

        with torch.no_grad():
            for x, padding_mask, answers in val_loader:
                x = x.to(DEVICE)
                padding_mask = padding_mask.to(DEVICE)

                prediction_scores = model(
                    x,
                    padding_mask,
                )

                binding_probability = torch.softmax(
                    prediction_scores,
                    dim=1,
                )[:, 1]

                prediction = prediction_scores.argmax(
                    dim=1
                )

                true_labels.extend(
                    answers.tolist()
                )

                predicted_labels.extend(
                    prediction.cpu().tolist()
                )

                probabilities.extend(
                    binding_probability.cpu().tolist()
                )

        auroc = roc_auc_score(
            true_labels,
            probabilities,
        )

        auprc = average_precision_score(
            true_labels,
            probabilities,
        )

        history["train_loss"].append(
            training_loss
        )

        history["val_auroc"].append(
            auroc
        )

        history["val_auprc"].append(
            auprc
        )

        print(
            f"{tokenizer_spec.name:20s} | "
            f"epoch {epoch}/{epochs} | "
            f"loss={training_loss:.4f} | "
            f"AUROC={auroc:.4f}"
        )

    training_time = (
        time.time()
        - start_time
    )

    metrics = {
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels,
        ),
        "precision": precision_score(
            true_labels,
            predicted_labels,
            zero_division=0,
        ),
        "recall": recall_score(
            true_labels,
            predicted_labels,
            zero_division=0,
        ),
        "f1": f1_score(
            true_labels,
            predicted_labels,
            zero_division=0,
        ),
        "auroc": roc_auc_score(
            true_labels,
            probabilities,
        ),
        "auprc": average_precision_score(
            true_labels,
            probabilities,
        ),
        "training_time_seconds": training_time,
    }

    evaluation = {
        "true": true_labels,
        "predicted": predicted_labels,
        "probability": probabilities,
    }

    return model, history, metrics, evaluation


def run_all_tokenizers(
    tokenizer_specs,
    model_class,
    sequences,
    labels,
    **training_settings,
):
    """Train one fresh model for every tokenizer."""

    results = {}

    for name, tokenizer_spec in tokenizer_specs.items():
        print()
        print("---", name, "---")

        model, history, metrics, evaluation = train_transformer(
            tokenizer_spec=tokenizer_spec,
            model_class=model_class,
            sequences=sequences,
            labels=labels,
            **training_settings,
        )

        results[name] = {
            "model": model,
            "history": history,
            "metrics": metrics,
            "evaluation": evaluation,
        }

    return results

# 2. Load the DNA

`load_dataset(...)` reads and cleans the prepared dataset.

We convert the returned lists to a DataFrame only so the first examples are easy to inspect.

In [ ]:
# ▶️ RUN

sequences, labels = load_dataset(DATA_DIR)

data = pd.DataFrame({
    "sequence": sequences,
    "label": labels,
})

data["label_name"] = data["label"].map({
    0: "Background",
    1: "Binding",
})

data.head()

# 3. Five ways to represent the same DNA

A **tokenizer** decides what counts as one input unit.

```mermaid
flowchart TD
    A["Raw DNA<br/>200 bases"] --> B["Single nucleotide<br/>1 base = 1 token"]
    A --> C["One-hot<br/>1 base = 1 vector"]
    A --> D["Overlapping 6-mer<br/>stride 1"]
    A --> E["Non-overlapping 6-mer<br/>stride 6"]
    A --> F["BPE<br/>learned chunks"]
```

The biological sequence stays the same. Only its numerical representation changes.

## Function: `build_tokenizers(...)`

This helper creates all five representations.

Its important input is:

- `train_sequences` → BPE learns its merge rules **only from the training split**.

`bpe_merges=80` means:

> Allow BPE to learn up to 80 common pair-merging rules.

In [ ]:
# ▶️ RUN — build tokenizers from training DNA only

training_sequences = get_training_sequences(
    sequences,
    labels,
)

tokenizers, bpe = build_tokenizers(
    training_sequences,
    bpe_merges=80,
)

In [ ]:
# ▶️ RUN — inspect how the same DNA changes

example_dna = sequences[0]

representation_rows = []

for name, tokenizer_spec in tokenizers.items():

    tokens = tokenizer_spec.show_tokens(
        example_dna
    )

    representation_rows.append({
        "tokenizer": name,
        "first_tokens": str(tokens[:5]),
        "token_count": len(tokens),
    })

representation_table = pd.DataFrame(
    representation_rows
)

representation_table

### Why use pandas here?

We have five representations with the same three pieces of information.

A table lets us compare them side-by-side without teaching a new pandas operation.

In [ ]:
# ▶️ RUN — token count graph

representation_table.plot(
    x="tokenizer",
    y="token_count",
    kind="bar",
    legend=False,
)

plt.ylabel("Tokens for one 200-bp sequence")
plt.title("Same DNA, different numbers of tokens")
plt.xticks(rotation=30, ha="right")
plt.show()

**Question this graph answers:**

> How much input does the Transformer have to process for each representation?

Attention compares tokens to other tokens, so token count can strongly affect compute cost.

# 4. Attention: the key Transformer idea

We will not implement the matrix equations ourselves, but we should understand the concept.

- **Query (Q):** What am I looking for?
- **Key (K):** What information do I offer?
- **Value (V):** What information should I pass along?

```mermaid
flowchart TD
    A["Input token vectors"] --> B["Queries Q"]
    A --> C["Keys K"]
    A --> D["Values V"]
    B --> E["Compare Q with K"]
    C --> E
    E --> F["Attention weights"]
    D --> G["Mix useful information"]
    F --> G
```

PyTorch's `nn.TransformerEncoderLayer` performs this standard operation for us.

# 5. The custom Transformer architecture

```mermaid
flowchart TD
    A["Tokenized DNA"] --> B{"Input type?"}
    B -->|"Token IDs"| C["nn.Embedding"]
    B -->|"One-hot"| D["nn.Linear projection"]
    C --> E["+ positional embedding"]
    D --> E
    E --> F["nn.TransformerEncoder"]
    F --> G["Average real token vectors"]
    G --> H["nn.Linear"]
    H --> I["Background / Binding"]
```

## Important PyTorch functions before you see the code

### `nn.Embedding(vocab_size, d_model)`

Converts an integer token ID into a learned vector.

- `vocab_size` → how many possible token IDs exist.
- `d_model` → length of the vector used inside the Transformer.

### `nn.Linear(4, d_model)`

One-hot inputs are already numerical vectors, so we use a linear layer to project their four numbers into the same `d_model` size.

### `nn.TransformerEncoderLayer(...)`

Creates **one standard Transformer block**.

- `d_model` → token-vector size.
- `nhead` → number of attention heads.
- `dim_feedforward=4*d_model` → size of the block's internal feed-forward network.
- `batch_first=True` → tensors are organized as batch → tokens → features.

### `nn.TransformerEncoder(layer, num_layers=...)`

Repeats that block several times.

In [ ]:
# 👀 READ — the model architecture

class DNAClassifier(nn.Module):

    def __init__(
        self,
        input_kind,
        vocab_size,
        max_length,
        d_model=64,
        nhead=4,
        layers=2,
    ):
        super().__init__()

        # Part 1: convert input into d_model-sized vectors.
        if input_kind == "one_hot":
            self.input_layer = nn.Linear(
                4,
                d_model,
            )
        else:
            self.input_layer = nn.Embedding(
                vocab_size,
                d_model,
                padding_idx=0,
            )

        # Part 2: learn where each token appears.
        self.position = nn.Embedding(
            max_length,
            d_model,
        )

        # Part 3: create one standard Transformer block.
        transformer_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4 * d_model,
            batch_first=True,
        )

        # Part 4: stack several Transformer blocks.
        self.transformer = nn.TransformerEncoder(
            transformer_layer,
            num_layers=layers,
        )

        # Part 5: convert sequence summary into two scores.
        self.classifier = nn.Linear(
            d_model,
            2,
        )

    def forward(self, x, padding_mask):

        # Convert tokens into vectors.
        x = self.input_layer(x)

        # Add information about token position.
        positions = torch.arange(
            x.shape[1],
            device=x.device,
        )

        x = x + self.position(positions)

        # Contextualize tokens with attention.
        x = self.transformer(
            x,
            src_key_padding_mask=padding_mask,
        )

        # Average real tokens while ignoring padding.
        keep = (~padding_mask).unsqueeze(-1)

        dna_summary = (
            (x * keep).sum(dim=1)
            / keep.sum(dim=1)
        )

        # Two final class scores.
        return self.classifier(
            dna_summary
        )

## Read the code as five model parts

```text
input_layer   → What token is this?
position      → Where is this token?
transformer   → Which other tokens matter?
dna_summary   → What does the sequence mean overall?
classifier    → Binding or Background?
```

If you understand those five lines conceptually, you understand the architecture.

In [ ]:
# ▶️ RUN — build one example model

example_spec = tokenizers["overlap_6mer"]

example_model = DNAClassifier(
    input_kind=example_spec.input_kind,
    vocab_size=example_spec.vocab_size,
    max_length=example_spec.max_length,
    d_model=64,
    nhead=4,
    layers=2,
)

print(example_model)

# 6. Training the custom Transformer

```mermaid
flowchart LR
    A["Tokenized batch"] --> B["Custom Transformer"]
    B --> C["Prediction scores"]
    C --> D["Loss"]
    D --> E["Backpropagation"]
    E --> F["AdamW optimizer"]
    F --> A
```

The training loop is hidden because the main lesson is the architecture.

## Function: `train_transformer(...)`

Important inputs:

- `tokenizer_spec` → which DNA representation to use,
- `d_model` → token-vector width,
- `nhead` → number of attention heads,
- `layers` → number of Transformer blocks,
- `epochs`, `batch_size`, `learning_rate` → training settings.

It builds a **fresh model**, trains it, and returns metrics and predictions.

In [ ]:
# ✏️ CHANGE — first model settings

D_MODEL = 64
HEADS = 4
LAYERS = 2

EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 1e-3

In [ ]:
# ▶️ RUN — train one tokenizer first

model, history, metrics, evaluation = train_transformer(
    tokenizer_spec=tokenizers["overlap_6mer"],
    model_class=DNAClassifier,
    sequences=sequences,
    labels=labels,
    d_model=D_MODEL,
    nhead=HEADS,
    layers=LAYERS,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
)

In [ ]:
# ▶️ RUN — metric table

pd.DataFrame([
    metrics
]).round(3)

## Evaluation graph 1 — training loss

Loss answers:

> Is optimization moving in a useful direction?

In [ ]:
plt.plot(
    range(1, len(history["train_loss"]) + 1),
    history["train_loss"],
    marker="o",
)

plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Custom Transformer training loss")
plt.show()

## Evaluation graph 2 — confusion matrix

This shows **which class errors** the model makes, not just how many.

In [ ]:
cm = confusion_matrix(
    evaluation["true"],
    evaluation["predicted"],
)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Background", "Binding"],
).plot()

plt.title("Custom Transformer confusion matrix")
plt.show()

## Evaluation graph 3 — ROC curve

In [ ]:
false_positive_rate, true_positive_rate, _ = roc_curve(
    evaluation["true"],
    evaluation["probability"],
)

plt.plot(
    false_positive_rate,
    true_positive_rate,
    label=f"AUROC = {metrics['auroc']:.3f}",
)

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve")
plt.legend()
plt.show()

## Evaluation graph 4 — precision–recall curve

In [ ]:
precision, recall, _ = precision_recall_curve(
    evaluation["true"],
    evaluation["probability"],
)

plt.plot(
    recall,
    precision,
    label=f"AUPRC = {metrics['auprc']:.3f}",
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall curve")
plt.legend()
plt.show()

# 7. Compare all five tokenizers

## Function: `run_all_tokenizers(...)`

This is a convenience function.

It repeats:

```text
choose tokenizer
↓
build a fresh Transformer
↓
train
↓
save result in memory
↓
move to next tokenizer
```

Every tokenizer receives the same model/training settings so the comparison is easier to interpret.

In [ ]:
# ▶️ RUN — train all five representations

all_results = run_all_tokenizers(
    tokenizer_specs=tokenizers,
    model_class=DNAClassifier,
    sequences=sequences,
    labels=labels,
    d_model=D_MODEL,
    nhead=HEADS,
    layers=LAYERS,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
)

In [ ]:
# ▶️ RUN — pandas makes the comparison compact

comparison_rows = []

for name, result in all_results.items():

    tokenizer_spec = tokenizers[name]

    comparison_rows.append({
        "tokenizer": name,
        "AUROC": result["metrics"]["auroc"],
        "AUPRC": result["metrics"]["auprc"],
        "accuracy": result["metrics"]["accuracy"],
        "training_seconds": result["metrics"]["training_time_seconds"],
        "tokens_in_example": len(
            tokenizer_spec.show_tokens(example_dna)
        ),
    })

comparison = pd.DataFrame(
    comparison_rows
)

comparison.round(3)

## Comparison graph — biological performance

In [ ]:
comparison.plot(
    x="tokenizer",
    y="AUROC",
    kind="bar",
    legend=False,
)

plt.ylim(0, 1)
plt.ylabel("Validation AUROC")
plt.title("Which DNA representation performed best?")
plt.xticks(rotation=30, ha="right")
plt.show()

## Comparison graph — computational cost

Training time is not a biological metric.

It answers a different question:

> Which representation is more expensive to process?

In [ ]:
comparison.plot(
    x="tokenizer",
    y="training_seconds",
    kind="bar",
    legend=False,
)

plt.ylabel("Training time (seconds)")
plt.title("Which representation took longer?")
plt.xticks(rotation=30, ha="right")
plt.show()

# ✅ Group B summary

```mermaid
flowchart LR
    A["DNA"] --> B["Tokenizer"]
    B --> C["Input vectors"]
    C --> D["+ Position"]
    D --> E["PyTorch Transformer Encoder"]
    E --> F["Sequence summary"]
    F --> G["Classifier"]
    G --> H["Confusion matrix / ROC / PR"]
    H --> I["Compare tokenizers"]
```

You are ready for Notebook 3B if you can explain what each box contributes.

# 🎨 Optional Visualization Playground

This section is **optional**. Nothing below is required to finish the notebook.

Use it when you want to ask your own question about a variable you created earlier.

```mermaid
flowchart LR
    A["Choose a variable"] --> B["Choose a term / column"]
    B --> C{"What do you want to see?"}
    C -->|"Counts / categories"| D["Bar plot"]
    C -->|"Distribution"| E["Histogram"]
    C -->|"Change across epochs"| F["Line plot"]
    C -->|"Relationship between numbers"| G["Scatter plot"]
```

## Two plotting patterns to remember

### One term / column

```python
VARIABLE["TERM"].plot(kind="hist")
```

Read it as:

> From this variable, choose this term, then plot it.

### Two terms / columns

```python
VARIABLE.plot(
    x="TERM1",
    y="TERM2",
    kind="scatter",
)
```

Read it as:

> Use `TERM1` for the x-axis and `TERM2` for the y-axis.

Useful `kind=` choices:

| `kind` | Good for |
|---|---|
| `"bar"` | comparing categories |
| `"hist"` | seeing a distribution |
| `"line"` | following change across epochs |
| `"scatter"` | comparing two numerical values |
| `"box"` | comparing distributions between groups |

If you forget what terms exist inside a pandas table, run:

```python
VARIABLE.columns.tolist()
```

## Important variables from Notebook 2

| Variable | What it contains | Terms you can explore |
|---|---|---|
| `data` | the DNA dataset | `sequence`, `label`, `label_name` |
| `representation_table` | how one DNA sequence looks under each tokenizer | `tokenizer`, `first_tokens`, `token_count` |
| `history_df` | training of the first custom Transformer | `epoch`, `train_loss`, `val_auroc`, `val_auprc` |
| `evaluation_df` | predictions from the first custom Transformer | `true`, `predicted`, `probability`, `correct` |
| `comparison` | all five tokenizer experiments | `tokenizer`, `AUROC`, `AUPRC`, `accuracy`, `training_seconds`, `tokens_in_example` |

### Questions you could visualize

- Which tokenizer creates the most tokens?
- Which tokenizer has the best AUROC or AUPRC?
- Which tokenizer trains fastest?
- Is token count related to training time?
- Is token count related to model performance?
- How did the first model improve across epochs?

In [ ]:
# ▶️ OPTIONAL — create visualization-friendly tables

history_df = pd.DataFrame(history)
history_df["epoch"] = range(1, len(history_df) + 1)

evaluation_df = pd.DataFrame(evaluation)
evaluation_df["correct"] = (
    evaluation_df["true"] == evaluation_df["predicted"]
)

print("data terms:", data.columns.tolist())
print("representation_table terms:", representation_table.columns.tolist())
print("history_df terms:", history_df.columns.tolist())
print("evaluation_df terms:", evaluation_df.columns.tolist())
print("comparison terms:", comparison.columns.tolist())

## Copy a visualization recipe and change the terms

### Tokenization

```python
representation_table.plot(
    x="tokenizer",
    y="token_count",
    kind="bar",
)
```

### Training history

```python
history_df.plot(
    x="epoch",
    y=["val_auroc", "val_auprc"],
    kind="line",
    marker="o",
)
```

### Compare tokenizer performance

```python
comparison.plot(
    x="tokenizer",
    y=["AUROC", "AUPRC"],
    kind="bar",
)
```

Try changing `y=` to:

```python
"training_seconds"
```

### Ask whether token count affects computation

```python
comparison.plot(
    x="tokens_in_example",
    y="training_seconds",
    kind="scatter",
)
```

### Ask whether token count relates to performance

```python
comparison.plot(
    x="tokens_in_example",
    y="AUROC",
    kind="scatter",
)
```

### Prediction confidence

```python
evaluation_df["probability"].plot(kind="hist", bins=20)
```

**Student challenge:** make one scientific graph and one computational graph. Are they telling the same story?